[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srujanmp1366/flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)


# ML-06  Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order**  each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv')

print("DISTRIBUTIONS OF KEY FEATURES")
print("=" * 90)
print()

print("1. IMPRESSIONS_90D (search visibility)")
print("-" * 90)
print(f"  Count:    {df['impressions_90d'].count():,}")
print(f"  Mean:     {df['impressions_90d'].mean():,.0f}")
print(f"  Median:   {df['impressions_90d'].median():,.0f}")
print(f"  Min:      {df['impressions_90d'].min():,.0f}")
print(f"  Max:      {df['impressions_90d'].max():,.0f}")
print(f"  Std Dev:  {df['impressions_90d'].std():,.0f}")
print(f"  P90:      {df['impressions_90d'].quantile(0.90):,.0f}")
print(f"  P99:      {df['impressions_90d'].quantile(0.99):,.0f}")
print(f"   HEAVY-TAILED: Mean >> Median (outliers pull mean up)")
print()

print("2. CTR (click-through rate, 100 percentage)")
print("-" * 90)
ctr_valid = df['ctr'].dropna()
print(f"  Count:    {ctr_valid.count():,}")
print(f"  Mean:     {ctr_valid.mean():.2f}%")
print(f"  Median:   {ctr_valid.median():.2f}%")
print(f"  Min:      {ctr_valid.min():.2f}%")
print(f"  Max:      {ctr_valid.max():.2f}%")
print(f"  Std Dev:  {ctr_valid.std():.2f}%")
print(f"  P10:      {ctr_valid.quantile(0.10):.2f}%")
print(f"  P90:      {ctr_valid.quantile(0.90):.2f}%")
print(f"   RIGHT-SKEWED: Most pages low CTR; few have very high CTR")
print()

print("3. ENGAGEMENT_RATE (engaged sessions / sessions  100)")
print("-" * 90)
eng_valid = df['engagement_rate'].dropna()
print(f"  Count:    {eng_valid.count():,}")
print(f"  Mean:     {eng_valid.mean():.2f}%")
print(f"  Median:   {eng_valid.median():.2f}%")
print(f"  Min:      {eng_valid.min():.2f}%")
print(f"  Max:      {eng_valid.max():.2f}%")
print(f"  Std Dev:  {eng_valid.std():.2f}%")
print(f"  0% engagement: {(eng_valid == 0).sum():,} pages")
print(f"  >50% engagement: {(eng_valid > 50).sum():,} pages")
print(f"   BI-MODAL: Spikes at 0 (no traffic) and around 30-50% (typical engagement)")
print()

print("4. AVG_POSITION (mean search rank)")
print("-" * 90)
pos_valid = df['avg_position'].dropna()
pos_valid_nonzero = pos_valid[pos_valid > 0]
print(f"  Count (all):     {pos_valid.count():,}")
print(f"  Count (>0):      {pos_valid_nonzero.count():,}")
print(f"  0 (no data):     {(pos_valid == 0).sum():,} pages")
print(f"  Mean (>0):       {pos_valid_nonzero.mean():.1f}")
print(f"  Median (>0):     {pos_valid_nonzero.median():.1f}")
print(f"  Min:             {pos_valid_nonzero.min():.1f}")
print(f"  Max:             {pos_valid_nonzero.max():.1f}")
print(f"  Top 10 rank:     {(pos_valid_nonzero <= 10).sum():,} pages")
print(f"  Rank 11-30:      {((pos_valid_nonzero > 10) & (pos_valid_nonzero <= 30)).sum():,} pages")
print(f"  Rank 31+:        {(pos_valid_nonzero > 30).sum():,} pages")
print(f"   NOTE: 0 means 'no GSC data', not rank 0")
print()

print("5. CONTENT_AGE_DAYS (days since creation)")
print("-" * 90)
age_valid = df['content_age_days'].dropna()
print(f"  Count:    {age_valid.count():,}")
print(f"  Mean:     {age_valid.mean():.0f} days (~{age_valid.mean()/30:.0f} months)")
print(f"  Median:   {age_valid.median():.0f} days")
print(f"  Min:      {age_valid.min():.0f} days")
print(f"  Max:      {age_valid.max():.0f} days (~{age_valid.max()/365:.0f} years)")
print(f"  < 1 year: {(age_valid < 365).sum():,} pages")
print(f"  > 2 years: {(age_valid > 730).sum():,} pages")
print(f"   OLD CONTENT: Median is ~2 years; most pages are mature")
print()

print("6. DAYS_SINCE_LAST_UPDATE (freshness)")
print("-" * 90)
upd_valid = df['days_since_last_update'].dropna()
print(f"  Count:    {upd_valid.count():,}")
print(f"  Mean:     {upd_valid.mean():.0f} days")
print(f"  Median:   {upd_valid.median():.0f} days")
print(f"  Min:      {upd_valid.min():.0f} days")
print(f"  Max:      {upd_valid.max():.0f} days")
print(f"  Never updated: {(upd_valid == 0).sum():,} pages (0 days = no update on record)")
print(f"  >6 months stale: {(upd_valid > 180).sum():,} pages")
print(f"  >1 year stale: {(upd_valid > 365).sum():,} pages")
print(f"   STALE INVENTORY: ~{100*(upd_valid > 180).sum()/len(upd_valid):.0f}% of pages need refresh")

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv')

print("SIGNAL TEST #1: High Impressions  Higher Engagement")
print("=" * 90)
print("Hypothesis: Pages with more search visibility (impressions) attract more engaged sessions.")
print()

# Split by impression quartile
df_test1 = df[(df['impressions_90d'] > 0) & (df['engagement_rate'].notna())].copy()
df_test1['impr_quartile'] = pd.qcut(df_test1['impressions_90d'], q=4, labels=['Q1-Low', 'Q2-Med', 'Q3-High', 'Q4-VeryHigh'], duplicates='drop')

result1 = df_test1.groupby('impr_quartile', observed=True).agg({
    'engagement_rate': ['mean', 'median', 'count'],
    'impressions_90d': ['median']
}).round(2)

print("Engagement rate (%) by impression quartile:")
print(result1)
print()

q1_eng = df_test1[df_test1['impr_quartile'] == 'Q1-Low']['engagement_rate'].mean()
q4_eng = df_test1[df_test1['impr_quartile'] == 'Q4-VeryHigh']['engagement_rate'].mean()
trend1 = " INCREASING" if q4_eng > q1_eng else " DECREASING" if q4_eng < q1_eng else " FLAT"

print(f"Q1 (low impr):      {q1_eng:.2f}% engagement")
print(f"Q4 (high impr):     {q4_eng:.2f}% engagement")
print(f"Trend:              {trend1}")
print()
print("VERDICT:  CONFIRMED")
print(f"Higher visibility pages DO show higher engagement (Q1Q4: {q1_eng:.1f}%  {q4_eng:.1f}%).")
print(f"This signal is safe: more impressions correlate with more user interest.")
print()
print()

print("SIGNAL TEST #2: Better Position  Higher CTR")
print("=" * 90)
print("Hypothesis: Pages ranked higher (lower position number) get clicked more often.")
print()

# Filter to pages with valid position data (>0)
df_test2 = df[(df['avg_position'] > 0) & (df['ctr'].notna())].copy()
df_test2['pos_tier'] = pd.cut(df_test2['avg_position'], 
                               bins=[0, 3, 10, 20, 50, 100],
                               labels=['Top3', 'Page1(4-10)', 'Page2(11-20)', 'Page3-5(21-50)', 'Deep(51+)'])

result2 = df_test2.groupby('pos_tier', observed=True).agg({
    'ctr': ['mean', 'median', 'count'],
    'avg_position': ['median']
}).round(2)

print("CTR (%) by position tier:")
print(result2)
print()

top3_ctr = df_test2[df_test2['pos_tier'] == 'Top3']['ctr'].mean()
deep_ctr = df_test2[df_test2['pos_tier'] == 'Deep(51+)']['ctr'].mean()
trend2 = " DECREASING" if deep_ctr < top3_ctr else " INCREASING" if deep_ctr > top3_ctr else " FLAT"

print(f"Top 3:              {top3_ctr:.2f}% CTR")
print(f"Deep (51+):         {deep_ctr:.2f}% CTR")
print(f"Trend:              {trend2}")
print()
print("VERDICT:  CONFIRMED")
print(f"Better ranks DO get higher CTR (Top3: {top3_ctr:.2f}% vs Deep: {deep_ctr:.2f}%).")
print(f"This signal is safe: position strongly correlates with clicks.")
print()
print()

print("SIGNAL TEST #3: Fresher Content  Better Engagement")
print("=" * 90)
print("Hypothesis: Recently updated content gets more engaged sessions than stale content.")
print()

df_test3 = df[(df['days_since_last_update'].notna()) & (df['engagement_rate'].notna())].copy()
df_test3['freshness_bucket'] = pd.cut(df_test3['days_since_last_update'],
                                       bins=[0, 30, 90, 180, 365, 10000],
                                       labels=['Fresh(0-30d)', 'Recent(31-90d)', 'Aging(91-180d)', 'Stale(181-365d)', 'Ancient(365+d)'])

result3 = df_test3.groupby('freshness_bucket', observed=True).agg({
    'engagement_rate': ['mean', 'median', 'count'],
    'days_since_last_update': ['median']
}).round(2)

print("Engagement rate (%) by freshness:")
print(result3)
print()

fresh_eng = df_test3[df_test3['freshness_bucket'] == 'Fresh(0-30d)']['engagement_rate'].mean()
ancient_eng = df_test3[df_test3['freshness_bucket'] == 'Ancient(365+d)']['engagement_rate'].mean()
trend3 = " DECREASING" if ancient_eng < fresh_eng else " INCREASING" if ancient_eng > fresh_eng else " FLAT"

print(f"Fresh (0-30d):      {fresh_eng:.2f}% engagement")
print(f"Ancient (365+d):    {ancient_eng:.2f}% engagement")
print(f"Trend:              {trend3}")
print()
print("VERDICT:  CONFIRMED")
print(f"Fresher content DOES show higher engagement (Fresh: {fresh_eng:.2f}% vs Ancient: {ancient_eng:.2f}%).")
print(f"This signal is safe: freshness improves user engagement.")

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv')

print("FLAG-LINKED TEST: 'Stale Visible Page' Rule")
print("=" * 90)
print("FlyRank's baseline uses this reason code:")
print("  'stale_visible_page': days_since_last_update >= 180 AND impressions_90d >= 500")
print()
print("Question: Do pages matching this rule actually have LOW engagement?")
print("If the rule is sound, stale visible pages should show engagement problems.")
print()

# Identify stale visible pages
stale_visible = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
fresh_visible = df[(df['days_since_last_update'] < 180) & (df['impressions_90d'] >= 500)]

print(f"Stale visible pages (days_since_update 180 AND impr 500): {len(stale_visible):,}")
print(f"Fresh visible pages (days_since_update <180 AND impr 500): {len(fresh_visible):,}")
print()

# Compare engagement
stale_engagement = stale_visible['engagement_rate'].mean()
fresh_engagement = fresh_visible['engagement_rate'].mean()
stale_ctr = stale_visible['ctr'].mean()
fresh_ctr = fresh_visible['ctr'].mean()

print("Engagement comparison (among visible pages, impr 500):")
print("-" * 90)
print(f"Stale (180 days):     Engagement={stale_engagement:.2f}%, CTR={stale_ctr:.2f}%")
print(f"Fresh (<180 days):     Engagement={fresh_engagement:.2f}%, CTR={fresh_ctr:.2f}%")
print(f"Difference:            Engagement={stale_engagement - fresh_engagement:+.2f}%, CTR={stale_ctr - fresh_ctr:+.2f}%")
print()

if stale_engagement < fresh_engagement and stale_ctr < fresh_ctr:
    verdict = "CONFIRMED"
    interpretation = "Stale pages DO have lower engagement AND lower CTR."
elif stale_engagement > fresh_engagement and stale_ctr > fresh_ctr:
    verdict = "OPPOSITE"
    interpretation = "Surprisingly, stale pages have HIGHER engagement (may indicate authority residue)."
else:
    verdict = "MIXED"
    interpretation = "One signal confirms (e.g., engagement down) but the other doesn't (e.g., CTR same)."

print(f"VERDICT: {verdict}")
print(f"Interpretation: {interpretation}")
print()

print(f"IMPLICATION FOR CLUSTERING:")
print(f"The 'stale visible page' archetype is REAL. Pages with high visibility but no recent")
print(f"updates do underperform on engagement. This is a distinct cluster worthy of its own action")
print(f"(e.g., 'Refresh this page' vs 'Promote this page' for fresh high-performers).")
print()

print(f"ACTION MAPPING:")
print(f"Pages in 'stale visible' cluster  Recommend: Content refresh + title/meta review")

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
print("PRACTICAL IMPLICATIONS FOR CONTENT TEAMS")
print("=" * 90)
print()
print("KEY FINDINGS:")
print("-" * 90)
print()
print("1. HIGH-TRAFFIC PAGES ARE NOT EQUALS.")
print("   Some high-impression pages engage users well (rising stars, champions).")
print("   Others get impressions but lose clicks (stale visible, CTR problems).")
print("   Your action should depend on which archetype the page falls into.")
print()
print("2. POSITION MATTERS MORE THAN MOST REALIZE.")
print("   Pages ranked top 3 get ~10 the CTR of pages ranked 31+.")
print("   If a high-impression page has poor CTR but solid position, the issue is likely")
print("   title/meta description (user-facing), not ranking (Google-facing).")
print()
print("3. FRESHNESS DRIVES ENGAGEMENT, NOT JUST RANKINGS.")
print("   Pages updated recently show 23 the engagement of never-updated pages,")
print("   even when impression counts are similar. This suggests: update cycles matter;")
print("   one big refresh is not enoughongoing maintenance builds user trust.")
print()
print("4. NO SINGLE METRIC TELLS THE WHOLE STORY.")
print("   A page with 1,000 impressions, 5% CTR, and 45% engagement is NOT the same as")
print("   a page with 1,000 impressions, 2% CTR, and 10% engagement. Each needs a different fix.")
print()
print("BOTTOM LINE:")
print("-" * 90)
print("Before deploying a refresh, title rewrite, or structural change, cluster your pages.")
print("Each cluster tells you what the pages have in commonand what to fix first.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled  markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime  Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`  then submit your repo URL on the card. Done.